In [ ]:
import os 
import random 
from pathlib import Path 
from collections import defaultdict 

In [ ]:
import numpy as np 
import pandas as pd
import torch  
import torch.nn as nn
import torch.nn.functional as F
import torchaudio  
from torch.utils.data import Dataset, DataLoader
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


# Configuration

In [ ]:
# Project folder: run the notebook from the repo root
BASE_DIR = Path.cwd()
ORIGINAL_DIR = BASE_DIR / "original"
RECORDING_DIR = BASE_DIR / "recording"
PREPROCESSED_DIR = BASE_DIR / "preprocessed_mels" 

TOTAL_SONGS = 10
SAMPLE_RATE = 22050 
CLIP_SECONDS = 5  # seconds
CLIP_SAMPLES = SAMPLE_RATE * CLIP_SECONDS  # samples

N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

BATCH_SIZE = 32
NUM_EPOCHS = 10 
LEARNING_RATE = 1e-3 

TRAIN_RATIO = 0.8
RANDOM_SEED = 42

DEVICE = ("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


# Find original songs and create label mapping

In [ ]:
SUPPORTED_AUDIO = {".wav", ".mp3", ".flac", ".ogg", ".m4a"} 
def find_audio_files(directory):
    return sorted(
        path for path in Path(directory).iterdir() if path.suffix.lower() in SUPPORTED_AUDIO
    )

original_paths = find_audio_files(ORIGINAL_DIR)
recording_paths = find_audio_files(RECORDING_DIR)

if len(original_paths) != 10: 
    raise ValueError(f"Expected 10 original audio files, found {len(original_paths)}")
if len(recording_paths) != 10:
    raise ValueError(f"Expected 10 recording audio files, found {len(recording_paths)}")

song_names = [path.stem for path in original_paths] 

song_to_label = {name: idx for idx, name in enumerate(song_names)} 
label_to_song = {idx: name for name, idx in song_to_label.items()}

print("Song to label mapping:")
for song, label in song_to_label.items():
    print(f"{song}: {label}")

# Load Audio

In [ ]:
def load_audio(path):
    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:  # If stereo, convert to mono
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)(waveform)
    return waveform # (1,n_samples)

# Build 5-second clip metadata

In [ ]:
def build_clip_metadata(paths): 
    """
        Each metadata item contains:
        {
            "path": audio file path,
            "start": first sample index,
            "label": song class,
            "clip_index": index inside the song
        }
    """
    all_clips_by_song = defaultdict(list) 
    for path in paths: 
        song_name = path.stem 
        label = song_to_label[song_name]
        waveform = load_audio(path)
        total_samples = waveform.shape[1]
        num_clips = total_samples // CLIP_SAMPLES

        for clip_index in range(num_clips):
            start = (clip_index * CLIP_SAMPLES) 
            all_clips_by_song[label].append({
                "path":str(path),
                "start": start,
                "label": label,
                "clip_index": clip_index
            })
    return all_clips_by_song 

# Train Test Split

In [ ]:
def train_test_split(clips_by_song,train_ratio = 0.8): 
    train_samples = [] 
    test_samples = [] 
    rng = random.Random(RANDOM_SEED) 

    for label,clips in clips_by_song.items(): 
        clips = clips.copy()

        rng.shuffle(clips)
        split_index = int(len(clips) * train_ratio) 
        train_clips = clips[:split_index]
        test_clips = clips[split_index:]
        train_samples.extend(train_clips)
        test_samples.extend(test_clips)
        
    rng.shuffle(train_samples)
    rng.shuffle(test_samples)
    return train_samples,test_samples

# Build original + Microphone Dataset

In [ ]:
original_clips_by_song = build_clip_metadata(original_paths)
recording_clips_by_song = build_clip_metadata(recording_paths)

orig_train,orig_test = train_test_split(original_clips_by_song,train_ratio=TRAIN_RATIO)
mic_train,mic_test = train_test_split(recording_clips_by_song,train_ratio=TRAIN_RATIO)

train_samples = orig_train + mic_train
test_samples = orig_test + mic_test

print(
    "\nOriginal train clips:",
    len(orig_train)
)

print(
    "Microphone train clips:",
    len(mic_train)
)

print(
    "Total train original clips:",
    len(train_samples)
)


print(
    "\nOriginal test clips:",
    len(orig_test)
)

print(
    "Microphone test clips:",
    len(mic_test)
)

print(
    "Total test clips:",
    len(test_samples)
)

# Audio Augmentation

In [ ]:
def add_noise(waveform,noise_std): 
    noise = torch.randn_like(waveform) * noise_std
    noise_added_waveform = waveform + noise
    return torch.clamp(noise_added_waveform, -1.0, 1.0)  

def change_pitch(waveform, sample_rate, n_steps): 
    # Change pitch by n_steps semitones
    return torchaudio.functional.pitch_shift(waveform, sample_rate, n_steps) 
def change_volume(waveform, gain_db): 
    # Change volume by gain_db decibels
    waveform =  waveform * (10 ** (gain_db / 20))
    return torch.clamp(waveform, -1.0, 1.0) 

def augment_audio(
    waveform,
    variant
):
    """
    0 = original

    Six augmentations:

    1 = weak noise
    2 = stronger noise

    3 = pitch +2 semitones
    4 = pitch -2 semitones

    5 = volume quieter (-6 dB)
    6 = volume louder (+6 dB)
    """

    if variant == 0:

        return waveform


    elif variant == 1:

        return add_noise(
            waveform,
            noise_std=0.005
        )


    elif variant == 2:

        return add_noise(
            waveform,
            noise_std=0.015
        )


    elif variant == 3:

        return change_pitch(
            waveform,
            sample_rate=SAMPLE_RATE,
            n_steps=2
        )


    elif variant == 4:

        return change_pitch(
            waveform,
            sample_rate=SAMPLE_RATE,
            n_steps=-2
        )


    elif variant == 5:

        return change_volume(
            waveform,
            gain_db=-6
        )


    elif variant == 6:

        return change_volume(
            waveform,
            gain_db=6
        )


    else:

        raise ValueError(
            f"Unknown augmentation variant: {variant}"
        )
    

# MEL Spectrogram

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS, 
    power=2.0
)
db_transform = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80) 

def waveform_to_mel(waveform): 
    mel = mel_transform(waveform) #(1,128,T)
    mel_db = db_transform(mel) #(1,128,T)
    mean = mel_db.mean(dim=-1, keepdim=True)
    std = mel_db.std(dim=-1, keepdim=True)
    mel_db_norm = (mel_db - mean) / (std + 1e-6)
    return mel_db_norm

# Precompute All Training Mel Spectrograms 

In [ ]:
def preprocess_and_save(samples,split_name,augment = False): 
    split_dir = PREPROCESSED_DIR / split_name 
    split_dir.mkdir(parents=True,exist_ok=True)
    save_samples = []

    # if mel spectrograms already exist, skip processing 
    save_samples = list(split_dir.glob("*.pt")) 
    if save_samples:
        print(f"Mel spectrograms for split '{split_name}' already exist. Skipping processing.")
        return save_samples

    samples = sorted(samples, key=lambda x:x["path"]) 
    current_path = None 
    current_waveform = None 
    num_variants = 7 if augment else 1 

    for i,sample in enumerate(samples): 
        path = sample["path"] 
        if path != current_path:
            current_waveform = load_audio(path) 
            current_path = path

        start = sample["start"]
        end = (start + CLIP_SAMPLES) 

        orginal_clip = current_waveform[:,start:end] # (1,clip_samples) 

        label = sample["label"]

        for variant in range(num_variants):
            if augment: 
                waveform = augment_audio(orginal_clip,variant)
            else:
                waveform = orginal_clip.clone()
            mel = waveform_to_mel(waveform) #(1,128,T)

            filename = (
                    f"sample_{i:05d}"
                    f"_variant_{variant}"
                    f"_label_{label}.pt"
            )
            save_path = split_dir / filename
            torch.save(
                {
                    "mel": mel,
                    "label": label
                },
                save_path)
            save_samples.append(save_path)
        if (i+1) % 20 == 0:
            print(f"Processed {i+1}/{len(samples)} samples for split '{split_name}'")
    print(f"Finished processing {len(samples)} samples for split '{split_name}'")
    return save_samples

# Run Preprocessing

In [ ]:
train_files = preprocess_and_save(train_samples,split_name="train",augment=True)
test_files = preprocess_and_save(test_samples,split_name="test",augment=False) 

print(f"Total train files: {len(train_files)}")
print(f"Total test files: {len(test_files)}")

# Custom Dataset

In [ ]:
class SongDataset(Dataset):
    def __init__(self,files): 
        self.files = files
    def __len__(self):
        return len(self.files)
    def __getitem__(self,idx):
        data = torch.load(self.files[idx],map_location=DEVICE)
        mel = data["mel"] 
        label = data["label"] 
        return mel,label

train_dataset = SongDataset(train_files)
test_dataset = SongDataset(test_files) 


# Custom Dataloader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

# CNN MODEL

In [ ]:
class songCNN(nn.Module): 
    def __init__(self,num_classes = 10): 
        super().__init__() 
        self.features = nn.Sequential( 
            # Shape of input: (B, 1, 128, T)

            # Convolution block 1 
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1), 
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Shape after block 1: (B, 16, 64, T/2)

            # Convolution block 2
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Shape after block 2: (B, 32, 32, T/4)

            # Convolution block 3
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), 
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), 
            # Shape after block 3: (B, 64, 16, T/8) 
            
            # Convolution block 4
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Shape after block 4: (B, 128, 16, T/8)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))  # Shape (B, 128, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(), # Shape (B, 128)
            nn.Dropout(0.7), 
            nn.Linear(128, num_classes) # Shape (B, num_classes)
        )

    def forward(self,x):
        x = self.features(x) # (B, 128, 16, T/8) <= (B, 1, 128, T)
        x = self.global_pool(x) # (B, 128, 1, 1) <= (B, 128, 16, T/8)
        x = self.classifier(x) # (B, num_classes) <= (B, 128, 1, 1)
        return x
model = songCNN(num_classes=TOTAL_SONGS).to(DEVICE) 

print("Model architecture:")
print(model)

# Loss and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE,weight_decay=1e-4)

# Training

In [ ]:
def train_model(model,train_loader,criterion,optimizer,epochs): 
    model.train() 

    for epoch in range(epochs): 
        running_loss = 0.0 
        correct = 0
        total = 0
        for batch_index,(x,y) in enumerate(train_loader): 
            x = x.to(DEVICE) 
            y = y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(x)  # (B, num_classes) <= (B, 1, 128, T) 

            loss = criterion(logits,y) 

            loss.backward() 

            optimizer.step() 

            running_loss += loss.item() 
            predictions = torch.argmax(logits,dim=1)  # (B,) <= (B, num_classes)
            correct += (predictions == y).sum().item() 

            total += y.size(0)

            if (batch_index + 1) % 10 == 0:
                print(
                    f"Epoch [{epoch+1}/{epochs}], "
                    f"Batch [{batch_index+1}/{len(train_loader)}], "
                    f"Loss: {running_loss / (batch_index + 1):.4f}, "
                    f"Accuracy: {correct / total:.4f}"
                )
        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = correct / total 
        print(
            f"Epoch [{epoch+1}/{epochs}] completed. "
            f"Loss: {epoch_loss:.4f}, "
            f"Accuracy: {epoch_accuracy:.4f}"
        )

train_model(model,train_loader,criterion,optimizer,NUM_EPOCHS)

# Evaluation

In [ ]:
def evaluate_model(model,test_loader): 
    model.eval() 
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)
            predictions = logits.argmax(dim=1)

            correct += (predictions == y).sum().item()
            total += y.size(0)

    test_accuracy = correct / total

    print(f"Correct: {correct}")
    print(f"Total: {total}")
    print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

evaluate_model(model,test_loader)

# Save model

In [ ]:
MODEL_PATH = BASE_DIR / "song_cnn_model.pth"
torch.save( {
    "model_state_dict": model.state_dict(),
    "label_to_song": label_to_song,
    "song_to_label": song_to_label
}, MODEL_PATH) 
print(f"Model saved to {MODEL_PATH}")

# Load Model

In [ ]:
load = torch.load(BASE_DIR/"song_cnn_model.pth") 
model = songCNN() 
model.load_state_dict(load["model_state_dict"])  
model = model.to(DEVICE)
label_to_song = load["label_to_song"]
song_to_label = load["song_to_label"]  


# Predict from an audio file

In [ ]:
def predict_song(model,audio_path): 
    model.eval() 
    waveform = load_audio(audio_path) # (1,n_samples) 
    total_samples = waveform.shape[1]
    all_probabilities = []

    with torch.no_grad():
        for start in range(0,total_samples,CLIP_SAMPLES): 
            end = start + CLIP_SAMPLES 
            clip = waveform[:,start:end]  # (1, clip_samples)
            if clip.shape[1] < CLIP_SAMPLES:
                padding_needed = CLIP_SAMPLES - clip.shape[1] 
                clip = F.pad(clip, (0, padding_needed)) # (1, clip_samples) 

            mel = waveform_to_mel(clip)  # (1, 128, T)
            mel = mel.unsqueeze(0).to(DEVICE)  # (1, 1, 128, T)
            mel = mel.to(DEVICE)
            logits = model(mel)  # (1, num_classes) 

            probabilites = F.softmax(logits,dim=1)  # (1, num_classes) 
            all_probabilities.append(probabilites.cpu()) 

    if len(all_probabilities) == 0:
        raise ValueError("No clips were processed. Check the audio file length.")

    all_probabilites = torch.stack(all_probabilities,dim=0)  # (num_clips, 1, num_classes) 

    all_predictions = torch.argmax(all_probabilites,dim=2)  # (num_clips, 1)

    average_probabilites = all_probabilites.mean(dim=0)  # (1, num_classes) 
    predicted_label = torch.argmax(average_probabilites,dim=1).item()  # scalar
    predicted_song = label_to_song[predicted_label] 
    confidence = average_probabilites[0, predicted_label].item()  
    
    print(f"All predictions: {all_predictions.squeeze().tolist()}")
    print(f"Predicted song: {predicted_song} with confidence: {confidence:.4f}") 
    return all_predictions, predicted_song, confidence 

# Live Recording 

In [ ]:
# ============================================================
# 21. LIVE MICROPHONE RECORDING ON MAC
# ============================================================

import sounddevice as sd
import soundfile as sf


LIVE_RECORDING_PATH = (
    BASE_DIR
    / "live_recording.wav"
)



def record_from_microphone(
    seconds=20,
    output_path=LIVE_RECORDING_PATH
):
    """
    Record audio directly from the Mac microphone.

    At 22050 Hz:

        20 seconds
        ->
        441000 samples

    sounddevice initially returns:

        [samples, channels]

    Example:

        [441000, 1]

    The WAV file is then saved and later loaded by
    the SAME predict_song() function used for normal files.
    """

    print(
        f"Recording for "
        f"{seconds} seconds..."
    )


    recording = sd.rec(
        int(
            seconds
            * SAMPLE_RATE
        ),

        samplerate=SAMPLE_RATE,

        channels=1,

        dtype="float32"
    )

    # Shape:
    #
    # [seconds * SAMPLE_RATE, 1]
    #
    # For 20 seconds:
    #
    # [441000, 1]


    sd.wait()


    sf.write(
        str(output_path),
        recording,
        SAMPLE_RATE
    )


    print(
        f"Recording saved to:\n"
        f"{output_path}"
    )


    return str(
        output_path
    )

# Live and Predict

In [ ]:
def predict_live(model,seconds=20):
    """
    Record from the microphone and predict the song.
    """

    audio_path = record_from_microphone(
        seconds=seconds
    )

    all_predictions, predicted_song, confidence = predict_song(
        model,
        audio_path
    )

    return all_predictions, predicted_song, confidence

In [ ]:
predict_live(model, seconds= 10)

In [ ]:
all_predictions, predicted_song, confidence = predict_song(
    model,
    BASE_DIR / "live_recording.wav"
)

In [ ]:
print(song_to_label[predicted_song])